# LLM 训练数据治理：PII 检测、来源追踪与可验证删除

**面试问题：训练语料含个人信息时，如何在入库前治理，并在删除请求后证明已移除？**

## 回答主线

1. 数据治理从采集时记录来源、授权、用途、保留期和主体标识开始，不能只在训练前跑一次正则。
2. 入库门禁应检测和最小化 PII，对必要字段做不可逆占位或受控 Token 化。
3. 每条训练样本必须能追踪到 shard、offset、版本和派生数据，否则删除请求无法落地。
4. 删除不是写一个 tombstone 就结束，还要重建受影响 shard、索引、缓存和后续 checkpoint。
5. 检测器会漏掉 Unicode 混淆和上下文型 PII，需要规范化、规则、模型和抽样审计组合。
6. 最终输出应是删除证据包而不是口头声明。

## 真实案例

八条脱敏客服训练记录来自工单、公开 FAQ 和未授权上传，包含邮箱、手机号、证件号及 Unicode 全角邮箱。主体 U-2 发起删除请求。我们实现入库治理、占位脱敏、shard provenance 索引和删除重建，并复现未规范化导致的漏检。案例使用仓库内生成的离线脱敏小数据，输出用于学习机制，不代表生产性能。

### 输入预览：八条记录的来源与授权

In [1]:
import hashlib  # 导入哈希函数以生成内容和 shard 证据。
import re  # 导入正则表达式以检测常见 PII。
import unicodedata  # 导入 Unicode 规范化以修复全角字符绕过。

records = [  # 构造八条带治理元数据的训练记录。
    {"id": "R1", "subject": "U-1", "source": "ticket", "consent": True, "retention": "2027-01", "text": "退款邮箱 alice@example.com"},  # 含邮箱且已授权。
    {"id": "R2", "subject": "U-2", "source": "ticket", "consent": True, "retention": "2027-01", "text": "联系电话 13800138000"},  # 删除主体的手机号记录。
    {"id": "R3", "subject": "U-2", "source": "ticket", "consent": True, "retention": "2027-01", "text": "证件号 11010519491231002X"},  # 删除主体的证件记录。
    {"id": "R4", "subject": None, "source": "public-faq", "consent": True, "retention": "permanent", "text": "退款通常3个工作日到账"},  # 无个人信息的公开内容。
    {"id": "R5", "subject": "U-3", "source": "upload", "consent": False, "retention": "none", "text": "内部通讯录 bob@corp.test"},  # 未授权上传应整体拒绝。
    {"id": "R6", "subject": "U-4", "source": "ticket", "consent": True, "retention": "2026-08", "text": "收件手机 13912345678"},  # 另一主体手机号。
    {"id": "R7", "subject": "U-5", "source": "ticket", "consent": True, "retention": "2027-01", "text": "全角邮箱 test＠example．com"},  # Unicode 混淆邮箱反例。
    {"id": "R8", "subject": None, "source": "public-faq", "consent": True, "retention": "permanent", "text": "发票需要合同号和税号"},  # 第二条公开内容。
]  # 完成治理数据集。
print("记录  subject  source      consent  text")  # 输出输入表头。
for record in records:  # 逐记录展示来源和授权。
    print(f"{record['id']}    {str(record['subject']):<7} {record['source']:<11} {str(record['consent']):<7} {record['text']}")  # 展示 PII 与未授权来源。

记录  subject  source      consent  text
R1    U-1     ticket      True    退款邮箱 alice@example.com
R2    U-2     ticket      True    联系电话 13800138000
R3    U-2     ticket      True    证件号 11010519491231002X
R4    None    public-faq  True    退款通常3个工作日到账
R5    U-3     upload      False   内部通讯录 bob@corp.test
R6    U-4     ticket      True    收件手机 13912345678
R7    U-5     ticket      True    全角邮箱 test＠example．com
R8    None    public-faq  True    发票需要合同号和税号


## Baseline 基线：只做简单 ASCII 正则后全部入库

In [2]:
ascii_patterns = {  # 定义最小 ASCII PII 模式。
    "email": re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"),  # 检测普通邮箱。
    "phone": re.compile(r"(?<!\d)1[3-9]\d{9}(?!\d)"),  # 检测中国大陆手机号。
    "id_card": re.compile(r"(?<!\d)\d{17}[0-9Xx](?!\d)"),  # 检测十八位证件号。
}  # 完成规则集合。
def detect_ascii(text):  # 在未规范化文本上运行模式。
    return {name: pattern.findall(text) for name, pattern in ascii_patterns.items() if pattern.search(text)}  # 返回命中的类别和片段。

baseline_rows = [{"id": record["id"], "hits": detect_ascii(record["text"]), "ingested": True} for record in records]  # 错误地忽略授权并全部入库。
print("记录  ASCII命中                                      入库")  # 输出基线表头。
for row in baseline_rows:  # 逐记录展示检测结果。
    print(f"{row['id']}    {str(row['hits']):<45} {row['ingested']}")  # 展示 R7 漏检和 R5 未授权仍入库。
baseline_pii_count = sum(bool(row["hits"]) for row in baseline_rows)  # 统计已发现 PII 记录数。
print(f"发现PII记录={baseline_pii_count}，R7漏检={not bool(baseline_rows[6]['hits'])}，未授权R5仍入库={baseline_rows[4]['ingested']}")  # 暴露两个治理缺口。

记录  ASCII命中                                      入库
R1    {'email': ['alice@example.com']}              True
R2    {'phone': ['13800138000']}                    True
R3    {'id_card': ['11010519491231002X']}           True
R4    {}                                            True
R5    {'email': ['bob@corp.test']}                  True
R6    {'phone': ['13912345678']}                    True
R7    {}                                            True
R8    {}                                            True
发现PII记录=5，R7漏检=True，未授权R5仍入库=True


### 核心实现：规范化、授权门禁与占位脱敏

In [3]:
def normalize_for_detection(text):  # 在检测前统一兼容字符。
    return unicodedata.normalize("NFKC", text)  # 将全角＠和．折叠为 ASCII 对应字符。

def redact(text, hits):  # 按固定类别把 PII 替换为不可逆占位符。
    redacted = text  # 从原文本开始处理。
    for category, values in hits.items():  # 逐类别遍历具体命中。
        for value in values:  # 替换当前 PII 片段。
            redacted = redacted.replace(value, f"<{category.upper()}>")  # 保留语义类别而移除原值。
    return redacted  # 返回训练可用的最小化文本。

def govern(record):  # 对单条记录执行来源、授权和 PII 门禁。
    normalized = normalize_for_detection(record["text"])  # 先规范化潜在混淆字符。
    hits = detect_ascii(normalized)  # 在规范化文本上检测 PII。
    if not record["consent"]:  # 未授权来源不可因脱敏而自动获得训练许可。
        return {"id": record["id"], "decision": "reject", "reason": "no-consent", "hits": hits, "text": None}  # 整条拒绝并保留审计原因。
    redacted = redact(normalized, hits)  # 对允许来源执行数据最小化。
    return {"id": record["id"], "decision": "accept", "reason": "consent-and-redacted", "hits": hits, "text": redacted}  # 返回可入库版本。

governed = [govern(record) for record in records]  # 处理全部八条记录。
print("记录  decision  PII类别                  训练文本")  # 输出治理结果表头。
for row in governed:  # 逐记录展示门禁和脱敏文本。
    print(f"{row['id']}    {row['decision']:<8} {str(sorted(row['hits'])):<22} {row['text']}")  # 展示 R5 拒绝和 R7 规范化命中。
print("R7 规范化前后：", records[6]["text"], "->", normalize_for_detection(records[6]["text"]))  # 展示 Unicode 修正过程。

记录  decision  PII类别                  训练文本
R1    accept   ['email']              退款邮箱 <EMAIL>
R2    accept   ['phone']              联系电话 <PHONE>
R3    accept   ['id_card']            证件号 <ID_CARD>
R4    accept   []                     退款通常3个工作日到账
R5    reject   ['email']              None
R6    accept   ['phone']              收件手机 <PHONE>
R7    accept   ['email']              全角邮箱 <EMAIL>
R8    accept   []                     发票需要合同号和税号
R7 规范化前后： 全角邮箱 test＠example．com -> 全角邮箱 test@example.com


## 结果解读：构建 Shard 与主体 Provenance 索引

In [4]:
accepted = [row for row in governed if row["decision"] == "accept"]  # 仅将通过门禁的记录进入训练集。
shards = {"shard-0": accepted[:4], "shard-1": accepted[4:]}  # 用两份小 shard 演示物理布局。
record_metadata = {record["id"]: record for record in records}  # 建立原始治理元数据索引。
provenance = {}  # 建立主体到训练物理位置的倒排索引。
shard_hashes = {}  # 保存删除前 shard 内容摘要。
for shard_id, rows in shards.items():  # 遍历每个训练 shard。
    payload = "\n".join(row["text"] for row in rows)  # 构造确定性的 shard 逻辑内容。
    shard_hashes[shard_id] = hashlib.sha256(payload.encode("utf-8")).hexdigest()[:12]  # 保存版本摘要。
    for offset, row in enumerate(rows):  # 记录每条样本在 shard 内的偏移。
        subject = record_metadata[row["id"]]["subject"]  # 读取数据主体标识。
        if subject is not None:  # 只有可识别主体需要删除索引。
            provenance.setdefault(subject, []).append({"record_id": row["id"], "shard": shard_id, "offset": offset, "shard_hash": shard_hashes[shard_id]})  # 保存精确物理位置。
print("Shard 摘要：", {shard: {"records": [row["id"] for row in rows], "hash": shard_hashes[shard]} for shard, rows in shards.items()})  # 展示训练布局。
print("U-2 provenance：", provenance["U-2"])  # 展示删除请求可定位 R2/R3。
print(f"原始8条 -> 接受{len(accepted)}条，未授权拒绝={len(records) - len(accepted)}条，所有已接收PII均为占位符。")  # 汇总治理结果。

Shard 摘要： {'shard-0': {'records': ['R1', 'R2', 'R3', 'R4'], 'hash': 'b35d893acf34'}, 'shard-1': {'records': ['R6', 'R7', 'R8'], 'hash': '9f9d24a6583b'}}
U-2 provenance： [{'record_id': 'R2', 'shard': 'shard-0', 'offset': 1, 'shard_hash': 'b35d893acf34'}, {'record_id': 'R3', 'shard': 'shard-0', 'offset': 2, 'shard_hash': 'b35d893acf34'}]
原始8条 -> 接受7条，未授权拒绝=1条，所有已接收PII均为占位符。


## 失败案例：只写 Tombstone 不重建训练 Shard

In [5]:
deletion_subject = "U-2"  # 定义数据主体删除请求。
tombstones = {item["record_id"] for item in provenance[deletion_subject]}  # 找到需要删除的记录 ID。
unsafe_searchable = [row["id"] for rows in shards.values() for row in rows if row["id"] in tombstones]  # 检查原 shard 中数据仍然存在。
rebuilt_shards = {shard_id: [row for row in rows if row["id"] not in tombstones] for shard_id, rows in shards.items()}  # 重建受影响 shard 并物理移除记录。
new_hashes = {shard_id: hashlib.sha256("\n".join(row["text"] for row in rows).encode("utf-8")).hexdigest()[:12] for shard_id, rows in rebuilt_shards.items()}  # 计算删除后版本摘要。
safe_searchable = [row["id"] for rows in rebuilt_shards.values() for row in rows if row["id"] in tombstones]  # 验证新 shard 不再包含目标记录。
changed_shards = [shard_id for shard_id in shards if shard_hashes[shard_id] != new_hashes[shard_id]]  # 定位真正需要下游失效的 shard。
print(f"只写tombstone后原Shard仍可找到={unsafe_searchable}")  # 展示删除声明与物理事实不一致。
print(f"重建后可找到={safe_searchable}，变更Shard={changed_shards}，新hash={new_hashes}")  # 展示物理删除证据。
print("修正策略：删除工作流必须级联到 shard、采样索引、缓存、派生语料和未来 checkpoint；已训练权重需按法规与风险流程另行处置。")  # 总结删除范围。

只写tombstone后原Shard仍可找到=['R2', 'R3']
重建后可找到=[]，变更Shard=['shard-0']，新hash={'shard-0': '5d51e881ca72', 'shard-1': '9f9d24a6583b'}
修正策略：删除工作流必须级联到 shard、采样索引、缓存、派生语料和未来 checkpoint；已训练权重需按法规与风险流程另行处置。


### 生产边界与删除证据包

In [6]:
deletion_evidence = {"request": "DEL-U2-77", "subject": deletion_subject, "records": sorted(tombstones), "affected_shards": changed_shards, "before_hash": {shard: shard_hashes[shard] for shard in changed_shards}, "after_hash": {shard: new_hashes[shard] for shard in changed_shards}, "residual_matches": len(safe_searchable)}  # 构造可审计删除证明。
print("删除证据包：", deletion_evidence)  # 展示请求、目标、版本变化和残留扫描。
print("生产替换点：真实治理需要 DLP/NER、合法基础与地域规则、加密 Token Vault、湖仓行级 lineage、派生资产目录、审批和不可篡改审计。")  # 明确正则与内存 shard 边界。

删除证据包： {'request': 'DEL-U2-77', 'subject': 'U-2', 'records': ['R2', 'R3'], 'affected_shards': ['shard-0'], 'before_hash': {'shard-0': 'b35d893acf34'}, 'after_hash': {'shard-0': '5d51e881ca72'}, 'residual_matches': 0}
生产替换点：真实治理需要 DLP/NER、合法基础与地域规则、加密 Token Vault、湖仓行级 lineage、派生资产目录、审批和不可篡改审计。


## 回归测试：最后只保护授权、Unicode、Lineage 与物理删除

In [7]:
assert governed[4]["decision"] == "reject" and governed[4]["reason"] == "no-consent"  # 验证未授权上传整体拒绝。
assert "email" in governed[6]["hits"] and governed[6]["text"] == "全角邮箱 <EMAIL>"  # 验证全角邮箱规范化后被检测和脱敏。
assert {item["record_id"] for item in provenance["U-2"]} == {"R2", "R3"}  # 验证主体索引定位两条派生样本。
assert unsafe_searchable == ["R2", "R3"] and safe_searchable == []  # 验证 Tombstone 不足而重建完成物理删除。
assert deletion_evidence["residual_matches"] == 0 and changed_shards == ["shard-0"]  # 验证残留扫描为零且只重建受影响 shard。
print("回归测试通过：授权拒绝、Unicode 检测、主体 Lineage、Tombstone 反例和物理删除均成立。")  # 用少量断言总结数据治理合同。

回归测试通过：授权拒绝、Unicode 检测、主体 Lineage、Tombstone 反例和物理删除均成立。
